# Infrastructure diagnostics — live public data

Owner: repository founder. Current workflow: 2026-09-20.

**Run All initiates one new bounded public Bitfinex R0 capture**, using the installed
collector and its existing canonical-prefix verification. Diagnose installed identity,
configuration, durable completion, native/canonical replay parity and observed quality.
Use the [offline version](infrastructure-diagnostics.ipynb) for synthetic fault injection
or inspection of a saved recording. No network or data fallback is performed.

Each attempt retains its source/configuration/logs under ignored `data/notebooks/`.
This foreground diagnostic does not deploy a service or qualify strategy health.
[Source contract](../docs/data/l3-collectors.md) · [Run and retain](notebook-operations.md#live-versions).

## 1. State the diagnostic and identify the installed candidate

Can the configured public capture finish durably and reproduce the same observations
through saved replay and published book admission? Match the complete installed runtime
inventory before reading any source result. Failure should identify an owning boundary.

In [ ]:
question = {
    "question": "Does one bounded source capture preserve durable input and replay parity?",
    "scope": "public Bitfinex R0, native recording and first 32 canonical positions",
    "limits": "no deployed watchdog, fault injection or execution qualification",
}

In [ ]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from importlib.metadata import distributions
from pathlib import Path
from uuid import uuid4

# third party library imports
import pandas as pd
from IPython.display import display

# project library imports
import market_data
from configuration import CaptureConfig
from market_data.adapters.sources import BookAdapters, SourceAdapters
from market_data.contracts import CaptureRequest
from market_data.datasets import DatasetPublication
from market_data.query import BookMarketRow, BookQuery
from market_data.replay import SavedCapture
from market_data.storage import RawStore
from research.exploration import (
    BookResearchInput,
    BookSelection,
    ResearchReader,
    ResearchSelection,
)
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/live/bitfinex-btc-usd.json").is_file(), "Use the repository root/kernel"
assert platform.python_version() == (root / ".python-version").read_text().strip()
assert "site-packages" in Path(market_data.__file__).resolve().parts
codec = DomainCodec()
pd.set_option("display.max_colwidth", None)

In [ ]:
source_root = root / "src"
installed_inputs = dict(RuntimeFiles.inputs())
source_files = {
    path.relative_to(source_root).as_posix(): path
    for package in RuntimeFiles.roots
    for path in (source_root / package).rglob("*")
    if path.is_file() and (path.suffix in {".py", ".json"} or path.name == "py.typed")
}
assert source_files.keys() == installed_inputs.keys(), "Installed runtime inventory differs"
for name, source in source_files.items():
    assert source.read_bytes() == installed_inputs[name].read_bytes(), (
        f"Stale installed file: {name}"
    )
notebook_path = Path(
    os.environ.get(
        "ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/infrastructure-diagnostics-live.ipynb"
    )
)
notebook = json.loads(notebook_path.read_text())
notebook_source = [(cell["cell_type"], "".join(cell["source"])) for cell in notebook["cells"]]
identity = {
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip(),
    "working_tree": "changed"
    if subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
    else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel", "pandas"}
    },
    "kernel": sys.executable,
    "package": market_data.__file__,
    "runtime_files": len(installed_inputs),
    "implementation_sha256": ImplementationIdentity.sha256(),
    "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "notebook_source_sha256": hashlib.sha256(
        json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
    ).hexdigest(),
    "attempt": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
}
display(pd.DataFrame(identity.items(), columns=["Identity", "Value"]))

## 2. Review bounds and run the existing collector

Review the displayed non-secret selection. Receipt lasts at most 20 seconds; the byte,
message, queue, idle and retry ceilings remain owned by `CaptureConfig`. The command has
an 80-second outer deadline; timeout kills/waits for the child and retains its prefix.
A nonzero exit or refused publication stops this notebook. Inspect the retained logs and
recording explicitly in the offline engineer version; never convert it into success.

In [ ]:
config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
if config.venue != "bitfinex" or config.channel != "R0" or config.duration_seconds > 20:
    raise ValueError("This notebook selects public Bitfinex R0 with a receive interval <=20s")
study = root / "data/notebooks/infrastructure-live" / str(uuid4())
study.mkdir(parents=True)
(study / "snapshots").mkdir()
(study / "raw").mkdir()
config_path = study / "capture-config.json"
config_path.write_bytes(codec.encode_json(config))
(study / "attempt.json").write_bytes(
    codec.encode_json({"identity": identity, "question": question})
)
display(
    pd.DataFrame(
        [
            {
                "venue": config.venue,
                "symbol": config.symbol,
                "channel": config.channel,
                "base": config.base_currency,
                "quote": config.quote_currency,
                "duration_seconds": config.duration_seconds,
                "messages_max": config.max_messages,
                "journal_bytes_max": config.max_bytes,
                "frame_bytes_max": config.max_frame_bytes,
                "retries": config.retries,
                "published_raw_positions_max": 32,
            }
        ]
    )
)
print("Retained attempt directory:", study)

In [ ]:
collector = Path(sys.executable).parent / "alpha-collect"
if not collector.is_file():
    raise FileNotFoundError(f"Installed collector is missing: {collector}")
command = [
    str(collector),
    "capture",
    "--config",
    str(config_path),
    "--output",
    str(study / "captures"),
    "--snapshot-directory",
    str(study / "snapshots"),
    "--snapshot-records",
    "32",
]
# Claim this attempt before any source effect; rerunning this cell requires new setup.
with (study / "command.json").open("xb") as command_file:
    command_file.write(codec.encode_json(command))
# Files retain partial diagnostics even if the child times out; no raw frames are displayed.
with (study / "capture.stdout").open("x") as stdout, (study / "capture.stderr").open("x") as stderr:
    capture = subprocess.run(
        command, cwd=study, stdout=stdout, stderr=stderr, timeout=80, check=False
    )
(study / "capture-exit.json").write_bytes(codec.encode_json({"exit_code": capture.returncode}))
if capture.returncode != 0:
    raise RuntimeError(
        f"Capture exit {capture.returncode}; inspect retained logs/prefix under {study}"
    )
summary = json.loads((study / "capture.stdout").read_text())
capture_directory = Path(summary["capture_directory"])
request = codec.decode_json((capture_directory / "request.json").read_bytes(), CaptureRequest)
assert request.config == config and request.identity == summary["request_identity"]
display(
    pd.DataFrame(
        [
            {
                name: summary[name]
                for name in (
                    "venue",
                    "symbol",
                    "messages",
                    "epochs",
                    "gaps",
                    "stop_reason",
                    "replay_parity",
                )
            }
        ]
    )
)

## 3. Verify native saved replay and canonical publication

The collector compares the full finite recording with saved native replay, plus the
selected canonical prefix with its live normalization. Reopen the saved recording again
and compare every native summary field. Then copy its verified raw object into a standalone
study store and republish the same prefix: the immutable snapshot identity must agree.

In [ ]:
replayed = SavedCapture.read(capture_directory, book_factory=BookAdapters.create).summary()
assert all(summary[name] == value for name, value in replayed.items())
assert summary["replay_parity"] is True and summary["canonical_parity"] is True
print("Complete native replay matched; live/canonical prefix parity matched.")

In [ ]:
# Copy the verified original into this study's raw/ store and publish a bounded prefix.
snapshot_id = DatasetPublication(SourceAdapters(), ImplementationIdentity.sha256()).publish_books(
    (capture_directory,),
    RawStore(study / "raw"),
    study / "snapshots",
    max_records=32,
    max_bytes=config.max_bytes,
)
# Query boundaries are microseconds; floor the start and ceil the excluded end.
# Original per-message integer nanoseconds remain untouched in the result.
epoch = datetime(1970, 1, 1, tzinfo=UTC)
start = epoch + timedelta(microseconds=summary["started_ns"] // 1000)
end = epoch + timedelta(microseconds=summary["finished_ns"] // 1000 + 1)
selection = ResearchSelection(
    inputs=(
        BookSelection(
            name="visible_depth",
            query=BookQuery(
                snapshot_id=snapshot_id,
                request_identity=request.identity,
                instrument=config.instrument_id,
                start=start,
                end=end,
                known_at=end,
                columns=("orders",),
                batch_size=32,
                max_bytes=config.max_bytes,
            ),
        ),
    )
)
selection_path = study / "selection.json"
selection_path.write_bytes(codec.encode_json(selection))
print("Snapshot:", snapshot_id)
print("Selection:", selection.identity)
print("UTC receipt interval [start, end):", start, end)

assert snapshot_id == summary["snapshot_identity"]
print("Reopened canonical snapshot identity matched the collector publication.")

## 4. Inspect quality and clocks without treating liveness as freshness

Read the same pinned book interface used by QR research. Inspect all prefix control
boundaries and the final native quality summary. Local receipt is integer UTC nanoseconds;
Bitfinex's selected flags supply no exchange event timestamp. Preserve absent source times.
Receipt spacing is not network latency, throughput qualification or persistence lag.
The end boundary makes current quality unavailable; it does not erase the saved sample.

In [ ]:
dataset = ResearchReader(
    study / "snapshots", RawStore(study / "raw"), readers=SourceAdapters()
).read(selection)
depth = dataset.inputs[0]
assert isinstance(depth, BookResearchInput)
rows = [row for batch in depth.batches for row in batch.rows]
market_rows = [row for row in rows if isinstance(row, BookMarketRow)]
controls = [row for row in rows if not isinstance(row, BookMarketRow)]
display(
    pd.DataFrame(
        [
            {
                "position": row.position,
                "received_ns": row.received_ns,
                "status": row.status,
                "reason": row.reason,
            }
            for row in controls
        ]
    )
)
display(
    pd.DataFrame(
        [
            {
                "market_states": len(market_rows),
                "controls": len(controls),
                "unknown_source_times": sum(row.source_time is None for row in market_rows),
                "epochs": summary["epochs"],
                "duplicates": summary["duplicates"],
                "gaps": len(summary["gaps"]),
                "retry_exhausted": summary["retry_exhausted"],
                "ended": summary["ended"],
                "terminal_quality": summary["quality"],
                "coverage_limits": depth.batches[0].reasons,
            }
        ]
    )
)

## 5. Interpret and retain the diagnostic

| Failure or observation | Functional owner / next check |
| --- | --- |
| Installed inventory mismatch | `runtime/` and the [kernel repair guide](notebook-operations.md#interactive-kernel-and-import-repair) |
| Access, idle, sequence or reconnect failure | `market_data/collection`, transport and the native adapter; inspect retained capture logs |
| Missing manifest or disk failure | `market_data/storage`; explicitly inspect the interrupted committed prefix offline |
| Native/canonical parity mismatch | `market_data/books`, replay and collection; retain `first-divergence.json` when emitted |
| Publication or query refusal | `market_data/datasets`, query and validation; preserve raw and snapshot bytes |
| Usable saved sample but missing empirical coverage | QR/source evidence remains unresolved; no backtest or execution acceptance |

The offline version owns deterministic failures and independent health-supervision exercises.
This live run uses the foreground publication path. The existing supervised collector is
raw-capture-only and refuses snapshot flags; do not imply this run tested its watchdog,
alert delivery or production availability. See [independent supervision](../docs/data/l3-collectors.md#independent-health-supervision).
No performance benchmark or remote alert is triggered here.

### Handoff through the current feature/model/economic boundaries

This source check produces a pinned **book** selection. The implemented trailing-return,
model-feature and spot-economic paths require eligible **trade** revisions; do not relabel
the book as those inputs. The offline [developer notebook](infrastructure-diagnostics.ipynb)
now executes the installed retained-feature/model and economic consumers, including
fresh-process model reload, complete pending-intent resume and vectorbt comparison.
The [QR notebook](research-foundations.ipynb) shows their connected synthetic study.
These are separate software evidence, not economic conclusions from this live capture.
No collector, publication or source-retention behavior changes in this live diagnostic.

In [ ]:
findings = {
    "snapshot": snapshot_id,
    "selection": selection.identity,
    "capture": str(capture_directory),
    "native_replay_parity": True,
    "canonical_prefix_parity": True,
    "market_states": len(market_rows),
    "status": "bounded source diagnostic completed; coverage remains unproven",
}
(study / "findings.json").write_bytes(codec.encode_json(findings))
print("Retained diagnostic:", study / "findings.json")
print("Offline engineer: set ALPHA_BITFINEX_CAPTURE_DIRECTORY to", capture_directory)
print("Offline QR: set ALPHA_RESEARCH_DATASET_DIRECTORY to", study)